Time Since Signup

In [1]:
import pandas as pd
import numpy as np

In [2]:
fraud = pd.read_csv("../data/processed/fraud_cleaned.csv")
credit = pd.read_csv("../data/processed/credit_cleaned.csv")

In [3]:
print(fraud.dtypes)

user_id                     int64
signup_time                object
purchase_time              object
purchase_value              int64
device_id                  object
source                     object
browser                    object
sex                        object
age                         int64
ip_address                  int64
class                       int64
lower_bound_ip_address    float64
upper_bound_ip_address    float64
country                    object
dtype: object


In [4]:
fraud["signup_time"] = pd.to_datetime(fraud["signup_time"])
fraud["purchase_time"] = pd.to_datetime(fraud["purchase_time"])

Time Since Signup

In [5]:
fraud["time_since_signup"] = (
    fraud["purchase_time"] -
    fraud["signup_time"]
).dt.total_seconds()

Hour

In [6]:
fraud["hour_of_day"] = fraud["purchase_time"].dt.hour

Day

In [7]:
fraud["day_of_week"] = fraud["purchase_time"].dt.day_name()

Transaction Count

In [8]:
fraud["transaction_count"] = (
    fraud.groupby("user_id")["user_id"]
         .transform("count")
)

Transaction Velocity

In [9]:
fraud.sort_values(
    ["user_id","purchase_time"],
    inplace=True
)

In [10]:
fraud["transaction_velocity"] = (
    fraud.groupby("user_id")
         ["purchase_time"]
         .diff()
         .dt.total_seconds()
)

One-hot Encoding

In [11]:
fraud.columns

Index(['user_id', 'signup_time', 'purchase_time', 'purchase_value',
       'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class',
       'lower_bound_ip_address', 'upper_bound_ip_address', 'country',
       'time_since_signup', 'hour_of_day', 'day_of_week', 'transaction_count',
       'transaction_velocity'],
      dtype='object')

In [12]:
print([col for col in fraud.columns if col.startswith("browser_")][:10])
print([col for col in fraud.columns if col.startswith("source_")][:10])
print([col for col in fraud.columns if col.startswith("sex_")][:10])

[]
[]
[]


In [13]:
print(fraud[[
    "time_since_signup",
    "transaction_count",
    "transaction_velocity"
]].head())

        time_since_signup  transaction_count  transaction_velocity
30049           3564984.0                  1                   NaN
95244          10039879.0                  1                   NaN
11606           6667201.0                  1                   NaN
101959          4631485.0                  1                   NaN
19600           3193080.0                  1                   NaN


In [14]:
print([col for col in fraud.columns if col.startswith("browser_")])
print([col for col in fraud.columns if col.startswith("source_")])
print([col for col in fraud.columns if col.startswith("sex_")])

[]
[]
[]


Scaling

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [16]:
numerical_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "transaction_count",
    "transaction_velocity"
]

fraud[numerical_cols] = scaler.fit_transform(
    fraud[numerical_cols]
)

c:\Users\hp-pc\fraud-detection\.venv\lib\site-packages\sklearn\utils\extmath.py:1144: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\hp-pc\fraud-detection\.venv\lib\site-packages\sklearn\utils\extmath.py:1149: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\hp-pc\fraud-detection\.venv\lib\site-packages\sklearn\utils\extmath.py:1169: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


Save Dataset

In [17]:
fraud.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)

In [18]:
credit.to_csv(
    "../data/processed/credit_processed.csv",
    index=False
)